In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [16]:
import requests

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"

response = requests.get(url)

if response.status_code == 200:
    # Open a local file in binary write mode
    with open("Chinook.db", "wb") as file:
        # Write the content of the response (the file) to the local file
        file.write(response.content)
    print("File downloaded and saved as Chinook.db")
else:
    print(f"Failed to download the file. Status code: {response.status_code}")

File downloaded and saved as Chinook.db


In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

Dialect: sqlite
Available tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [18]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [19]:
from langgraph.prebuilt import create_react_agent

system_prompt_korean = """
당신은 SQL 데이터베이스와 상호 작용하도록 설계된 에이전트입니다.
입력된 질문이 주어지면, 실행할 구문적으로 정확한 {dialect} 쿼리를 생성하고,
쿼리 결과를 확인한 다음 답변을 반환합니다. 사용자가 얻고자 하는 특정 예시의
개수를 명시하지 않는 한, 항상 쿼리의 결과를 최대 {top_k}개로 제한해야 합니다.

데이터베이스에서 가장 흥미로운 예시를 반환하기 위해 관련 열을 기준으로
결과를 정렬할 수 있습니다. 특정 테이블의 모든 열을 쿼리하지 말고,
주어진 질문에 관련된 열만 요청해야 합니다.

쿼리를 실행하기 전에 **반드시** 두 번 확인해야 합니다. 쿼리 실행 중
오류가 발생하면, 쿼리를 다시 작성하고 재시도하세요.

데이터베이스에 **어떤 DML 문**(INSERT, UPDATE, DELETE, DROP 등)도 만들지 마세요.

시작할 때, 쿼리할 수 있는 항목을 확인하기 위해 **항상** 데이터베이스의
테이블을 살펴보아야 합니다. 이 단계를 건너뛰지 마세요.

그 다음에는 가장 관련성이 높은 테이블의 스키마를 쿼리해야 합니다.
""".format(
    dialect=db.dialect,
    top_k=5,
)

agent = create_react_agent(
    llm,
    tools,
    prompt=system_prompt,
)

In [20]:
question_korean = "평균적으로 트랙 길이가 가장 긴 장르는 무엇입니까?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which genre on average has the longest tracks?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (04b17a8c-e01f-46f4-8250-032f314e3632)
 Call ID: 04b17a8c-e01f-46f4-8250-032f314e3632
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (8e72dca7-420b-4e62-a307-14540a436620)
 Call ID: 8e72dca7-420b-4e62-a307-14540a436620
  Args:
    table_names: Track, Genre
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("GenreId")
)

/*
3 rows